<a href="https://colab.research.google.com/github/DeliaRudy/Storytelling/blob/dev/business_story_telling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#!pip install google-cloud-texttospeech


In [ ]:
# @title
from google.colab import files
uploaded = files.upload()  # This will prompt you to select the file from your computer

In [ ]:
# @title
import os

# Replace 'path/to/your/credentials.json' with the actual file path
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = ""


# **Init**

This is the starting point, generating audio using the [Text to Speech API](https://) to listen to or download.

In [ ]:
from google.cloud import texttospeech

# Instantiate a client
client = texttospeech.TextToSpeechClient()

# Set the text input to be synthesized
synthesis_input = texttospeech.SynthesisInput(text="Audacieuse en 5 secondes, redéfini par Rouge!")

# Build the voice request, select the language code ("en-US") and the ssml voice gender
voice = texttospeech.VoiceSelectionParams(
    language_code="fr-FR",
    ssml_gender=texttospeech.SsmlVoiceGender.NEUTRAL
)

# Select the type of audio file you want returned
audio_config = texttospeech.AudioConfig(
    audio_encoding=texttospeech.AudioEncoding.MP3
)

# Perform the text-to-speech request on the text input with the selected voice parameters and audio file type
response = client.synthesize_speech(
    input=synthesis_input, voice=voice, audio_config=audio_config
)

# Write the binary audio content to a file
with open("output.mp3", "wb") as out:
    out.write(response.audio_content)
print("Audio content written to file 'output.mp3'")


In [ ]:
from IPython.display import Audio

Audio("output.mp3")

# Multi- Speaker Audio

Generate the marketing script by using a story telling approach through a multi speaker podcast.

In [ ]:
from google.cloud import texttospeech

# Instantiate a client
client = texttospeech.TextToSpeechClient()

# Define the script for each speaker
speaker_1_text = "She is so confident??"
speaker_2_text = "Need a power move?, Get Red by Rouge!"

# Function to generate and save speech for each speaker
def synthesize_speech(text, voice_name, filename):
    synthesis_input = texttospeech.SynthesisInput(text=text)

    # Select the voice
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name=voice_name  # Specific voice for distinction
    )

    # Audio config
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3
    )

    # Perform TTS
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    # Save to file
    with open(filename, "wb") as out:
        out.write(response.audio_content)
    print(f"Audio content written to file '{filename}'")

# Synthesize each speaker with a different voice
synthesize_speech(speaker_1_text, "en-US-Wavenet-D", "speaker1.mp3")  # Male voice
synthesize_speech(speaker_2_text, "en-US-Chirp3-HD-Kore", "speaker2.mp3")  # Female voice


In [ ]:
#!pip install pydub
from pydub import AudioSegment

speaker1 = AudioSegment.from_file("speaker1.mp3")
speaker2 = AudioSegment.from_file("speaker2.mp3")

podcast = speaker1 + AudioSegment.silent(duration=300) + speaker2
podcast.export("ai_consultant_podcast.mp3", format="mp3")
print("Combined podcast exported as 'ai_consultant_podcast.mp3'")


In [ ]:
# prompt: display the audio from the podcast

Audio("ai_consultant_podcast.mp3")


# Ads from a prompt



In [ ]:
# 🧠 Setup Gemini
import google.generativeai as genai

# ENTER YOUR GEMINI API KEY HERE
import google.generativeai as genai
from google.colab import _message
secret = _message.blocking_request('get_secret', request={'name': 'GOOGLE_API_KEY_1'}, timeout_sec=5)
#genai.configure(api_key=secret['data'])
# Check if the secret was retrieved successfully
if 'data' in secret:
  genai.configure(api_key=secret['data'])
else:
  print(f"Error retrieving API key: {secret.get('error')}")  # Print error details if available
  # Provide alternative API key configuration or error handling
  # For example, you can ask the user to enter the API key manually:
  api_key = input("Enter your Gemini API key: ")
  genai.configure(api_key=api_key)

# ✏️ Input your podcast prompt
user_prompt = input("Enter your podcast prompt: ")

# Enhance the prompt to guide Gemini
full_prompt = f"""
Using a Gen Z, business casual tone, generate a short 15-second podcast script between two speakers.
Respond in JSON format only using this structure:

{{
  "speakers": [
    {{
      "name": "Speaker 1",
      "voice": "en-US-Wavenet-D",
      "text": "..."
    }},
    {{
      "name": "Speaker 2",
      "voice": "en-US-Wavenet-F",
      "text": "..."
    }}
  ]
}}

Prompt context: {user_prompt}
"""

# Call Gemini
# The model ID should be in the format 'models/gemini-pro' or 'models/gemini-pro-vision'
model = genai.GenerativeModel("models/gemini-pro") # Changed model ID to the correct format

response = model.generate_content(full_prompt)

# Extract JSON from response
import json, re

try:
    json_str = re.search(r'\{.*\}', response.text, re.DOTALL).group()
    script_data = json.loads(json_str)
    print("✅ Script generated:\n", script_data)
except Exception as e:
    print("⚠️ Error parsing script:", e)
    print(response.text)